<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.6**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.6 Change**
> - **GPT-5-pro**: Uses GPT-5-pro for inference instead of GPT-4o
> - **v7.5 Corrections**: Applies attribute-only corrections at the end
> - **Same pipeline**: All other features from v7.1 maintained


In [ ]:
# ==== 1) Model Configuration for GPT-5-pro ====
import os
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-5-pro, gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    return s

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-5-pro (latest and most capable)
    "gpt-5-pro": "gpt-5-pro",
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-5-pro"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


## **2** | Environment Setup

### **2a** | API Key Setup
#### **2a.1** | Access
1. Click the 🔑 icon in the left sidebar
2. Add your OpenAI API key
3. Set `OPENAI_MODEL` to `gpt-5-pro` (or leave blank for default)

#### **2a.2** | API Keys in Google Colab
The notebook will automatically read your API key from the 🔑 panel.


In [ ]:
# ==== Cell 3.9 — Version banner & quick sanity =====
from pathlib import Path
import glob, sys

NOTEBOOK_VERSION = "v7.6"
print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"full_text_for_row defined: {hasattr(globals(), 'full_text_for_row')}")
print(f"OUTPUTS_DIR defined: {hasattr(globals(), 'OUTPUTS_DIR')}")
print(f"BATCH_INPUT_CSV: {globals().get('BATCH_INPUT_CSV', None)}")
print(f"MODEL_ID: {globals().get('MODEL_ID', 'NOT SET')}")


## **3** | The Data

This notebook will use the same data structure as v7.1 but with GPT-5-pro inference.


In [ ]:
# ==== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load the three required files - Updated for Colab
RUN_ROOT = Path('/content')
BATCH_INPUT_CSV = RUN_ROOT / "extracted/inputs-v7.1/Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "extracted/inputs-v7.1/Ground Truth Masterfile.csv"
MNPS_ROLES_CSV = RUN_ROOT / "extracted/inputs-v7.1/MNPS Roles.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV)
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV)

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")


## **4** | The Prompts

The notebook will use the same prompt structure as v7.1 but with GPT-5-pro for inference.


In [ ]:
# ==== Cell 15.0 — Role Confidence Output Schema (v6.0) =====
from pydantic import BaseModel, Field
from typing import List, Optional

class RoleConfidenceTable(BaseModel):
    """Schema for role confidence evaluation output."""
    role: str = Field(description="The MNPS role name")
    confidence: float = Field(description="Confidence score 0.0-1.0")
    reasoning: str = Field(description="Brief explanation of the confidence score")

class RoleConfidenceResponse(BaseModel):
    """Response containing role confidence evaluations."""
    evaluations: List[RoleConfidenceTable] = Field(description="List of role confidence evaluations")

print("✅ Role confidence schema defined")


In [ ]:
# ==== Cell 15.1 — Role Confidence Prompt Builder (v6.0) =====
def build_role_confidence_prompt(job_description: str, mnps_roles: List[str], ksacs: str) -> str:
    """Build prompt for role confidence evaluation."""
    roles_list = "\n".join([f"- {role}" for role in mnps_roles])
    
    prompt = f"""You are an expert job classification system for Metro Nashville Public Schools (MNPS).

Your task is to evaluate how well a job description matches each MNPS role based on the job attributes (Position Summary, Essential Functions, Work Experience, Education, Licenses and Certifications, Knowledge, Skills and Abilities).

**IMPORTANT**: Ignore the job title completely. Base your evaluation solely on the job attributes.

Available MNPS Roles:
{roles_list}

MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):
{ksacs}

Job Description:
{job_description}

For each MNPS role, provide:
1. A confidence score (0.0-1.0) indicating how well the job description matches the role
2. Brief reasoning for your confidence score

Return your response as a JSON object with the following structure:
{{
  "evaluations": [
    {{
      "role": "Role Name",
      "confidence": 0.85,
      "reasoning": "Brief explanation"
    }}
  ]
}}

Evaluate ALL roles listed above."""
    
    return prompt

print("✅ Role confidence prompt builder defined")


In [ ]:
# ==== Cell 15.2 — Role Confidence Shortlist (robust build + canonicalize + write; empty-safe) =====
import json
from openai import OpenAI

client = OpenAI()

def call_llm_json(prompt: str, model: str = None) -> dict:
    """Call OpenAI API with JSON response."""
    if model is None:
        model = MODEL_ID
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.2
    )
    
    return json.loads(response.choices[0].message.content)

# Get MNPS roles and KSACs
VALID_ROLES = roles_df['Role'].tolist()
KSACS_TEXT = """[KSACs content from the roles file]"""

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")
print(f"✅ Using model: {MODEL_ID}")


In [ ]:
# ==== Cell 16 — Batch Processing with GPT-5-pro =====
import time
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""
    
    # Get role confidence
    prompt = build_role_confidence_prompt(job_text, VALID_ROLES, KSACS_TEXT)
    
    try:
        response = call_llm_json(prompt, MODEL_ID)
        evaluations = response.get('evaluations', [])
        
        # Find best role
        best_role = max(evaluations, key=lambda x: x['confidence']) if evaluations else None
        
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': f"{best_role['role']} I" if best_role else 'Unknown',
            'major_role_group': best_role['role'] if best_role else 'Other',
            'minor_sub_group': 'I',  # Default to I, will be refined in v7.5 corrections
            'grouping_justification': best_role['reasoning'] if best_role else 'No match found',
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.1)  # Rate limiting

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")


## **5** | v7.5 Corrections Applied

Now apply the same corrections from v7.5 to the GPT-5-pro results:


In [ ]:
# ==== v7.5 Corrections for GPT-5-pro Results =====
import re
import numpy as np

# Load the GPT-5-pro results
preds = results_df.copy()
attrs = df.copy()

# Closed sets and normalization helpers
MAJOR_ALLOWED = [
    'Technician','Specialist','Analyst','Manager','Coordinator','Director','Other',
    'Teacher','Coach','Counselor','Clerical Support','Instructor','Driver'
]
MINOR_ALLOWED = ['I','II','III','Lead']

CANON_MINOR_MAP = {
    'i':'I','1':'I','one':'I','entry':'I',
    'ii':'II','2':'II','two':'II',
    'iii':'III','3':'III','three':'III',
    'lead':'Lead','iv':'III','4':'III'
}

SPECIALIST_FALLBACKS = [
    ('Teacher','classroom|lesson|instruction|teacher|students'),
    ('Coach','coach|instructional coach|plc|model lessons|co-teach'),
    ('Clerical Support','clerk|clerical|records|data entry|office support'),
    ('Counselor','counsel|social-emotional|guidance'),
    ('Manager','manage|supervise|budget|oversight|lead team|program manager'),
]

def normalize_minor(x: str) -> str:
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

print("✅ v7.5 correction functions defined")


In [ ]:
# ==== Apply v7.5 Corrections =====

# Build attribute-only text
ATTR_COLS = [
    'Position Summary','Essential Functions','Work Experience','Education',
    'Licenses and Certifications','Knowledge, Skills and Abilities'
]

text = (
    attrs['Position Summary'].fillna('') + ' ' +
    attrs['Essential Functions'].fillna('') + ' ' +
    attrs['Work Experience'].fillna('') + ' ' +
    attrs['Education'].fillna('') + ' ' +
    attrs['Licenses and Certifications'].fillna('') + ' ' +
    attrs['Knowledge, Skills and Abilities'].fillna('')
)

# Apply corrections
maj0 = preds.get('major_role_group', pd.Series(['Other']*len(preds)))
min0 = preds.get('minor_sub_group', pd.Series(['I']*len(preds)))

ref_major = []
for i, m in enumerate(maj0):
    proposed = str(m) if pd.notna(m) else 'Other'
    proposed = proposed if proposed in MAJOR_ALLOWED else 'Other'
    proposed = discourage_specialist(text.iloc[i], proposed)
    ref_major.append(proposed)

ref_minor = [normalize_minor(x) for x in min0]

# Create corrected results
corrected = preds.copy()
corrected['major_role_group'] = ref_major
corrected['minor_sub_group'] = ref_minor

# Save corrected results
corrected_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro_v75_corrected.csv"
corrected.to_csv(corrected_path, index=False)

print(f"✅ Applied v7.5 corrections")
print(f"✅ Saved corrected results to: {corrected_path}")


In [ ]:
# ==== Generate Summary Statistics =====

before_major = preds.get('major_role_group', pd.Series(['']*len(preds))).astype(str)
before_minor = preds.get('minor_sub_group', pd.Series(['']*len(preds))).astype(str)
after_major  = corrected['major_role_group'].astype(str)
after_minor  = corrected['minor_sub_group'].astype(str)

counts = pd.DataFrame({
    'key': ['rows','major_changed','minor_changed','specialist_after_count'],
    'value': [
        len(corrected),
        int((before_major!=after_major).sum()),
        int((before_minor!=after_minor).sum()),
        int((after_major=='Specialist').sum())
    ]
})

counts_path = OUTPUTS_DIR / "correction_counts_gpt5pro.csv"
counts.to_csv(counts_path, index=False)

# Show examples of changes
ex_idx = ((before_major!=after_major) | (before_minor!=after_minor)).to_numpy().nonzero()[0][:6]
examples = pd.DataFrame({
    'row': ex_idx,
    'job_title_original': preds['job_title_original'].iloc[ex_idx],
    'major_before': before_major.iloc[ex_idx],
    'major_after': after_major.iloc[ex_idx],
    'minor_before': before_minor.iloc[ex_idx],
    'minor_after': after_minor.iloc[ex_idx],
})

examples_path = OUTPUTS_DIR / "examples_gpt5pro.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(counts.to_string(index=False))

print("\n📝 Example Corrections:")
print(examples.to_string(index=False))

print(f"\n✅ Saved counts to: {counts_path}")
print(f"✅ Saved examples to: {examples_path}")


<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.6**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.6 Change**
> - **GPT-5-pro**: Uses GPT-5-pro for inference instead of GPT-4o
> - **v7.5 Corrections**: Applies attribute-only corrections at the end
> - **Same pipeline**: All other features from v7.1 maintained


In [ ]:
# ==== 1) Model Configuration for GPT-5-pro ====
import os
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-5-pro, gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    return s

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-5-pro (latest and most capable)
    "gpt-5-pro": "gpt-5-pro",
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-5-pro"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


## **2** | Environment Setup

### **2a** | API Key Setup
#### **2a.1** | Access
1. Click the 🔑 icon in the left sidebar
2. Add your OpenAI API key
3. Set `OPENAI_MODEL` to `gpt-5-pro` (or leave blank for default)

#### **2a.2** | API Keys in Google Colab
The notebook will automatically read your API key from the 🔑 panel.


In [ ]:
# ==== Cell 3.9 — Version banner & quick sanity =====
from pathlib import Path
import glob, sys

NOTEBOOK_VERSION = "v7.6"
print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"full_text_for_row defined: {hasattr(globals(), 'full_text_for_row')}")
print(f"OUTPUTS_DIR defined: {hasattr(globals(), 'OUTPUTS_DIR')}")
print(f"BATCH_INPUT_CSV: {globals().get('BATCH_INPUT_CSV', None)}")
print(f"MODEL_ID: {globals().get('MODEL_ID', 'NOT SET')}")


## **3** | The Data

This notebook will use the same data structure as v7.1 but with GPT-5-pro inference.


In [ ]:
# ==== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load the three required files
BATCH_INPUT_CSV = Path("extracted/inputs-v7.1/Sample JDs.csv")
GT_MASTERFILE_CSV = Path("extracted/inputs-v7.1/Ground Truth Masterfile.csv")
MNPS_ROLES_CSV = Path("extracted/inputs-v7.1/MNPS Roles.csv")

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV)
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV)

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")


## **4** | The Prompts

The notebook will use the same prompt structure as v7.1 but with GPT-5-pro for inference.


In [ ]:
# ==== Cell 15.0 — Role Confidence Output Schema (v6.0) =====
from pydantic import BaseModel, Field
from typing import List, Optional

class RoleConfidenceTable(BaseModel):
    """Schema for role confidence evaluation output."""
    role: str = Field(description="The MNPS role name")
    confidence: float = Field(description="Confidence score 0.0-1.0")
    reasoning: str = Field(description="Brief explanation of the confidence score")

class RoleConfidenceResponse(BaseModel):
    """Response containing role confidence evaluations."""
    evaluations: List[RoleConfidenceTable] = Field(description="List of role confidence evaluations")

print("✅ Role confidence schema defined")


In [ ]:
# ==== Cell 15.1 — Role Confidence Prompt Builder (v6.0) =====
def build_role_confidence_prompt(job_description: str, mnps_roles: List[str], ksacs: str) -> str:
    """Build prompt for role confidence evaluation."""
    roles_list = "\n".join([f"- {role}" for role in mnps_roles])
    
    prompt = f"""You are an expert job classification system for Metro Nashville Public Schools (MNPS).

Your task is to evaluate how well a job description matches each MNPS role based on the job attributes (Position Summary, Essential Functions, Work Experience, Education, Licenses and Certifications, Knowledge, Skills and Abilities).

**IMPORTANT**: Ignore the job title completely. Base your evaluation solely on the job attributes.

Available MNPS Roles:
{roles_list}

MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):
{ksacs}

Job Description:
{job_description}

For each MNPS role, provide:
1. A confidence score (0.0-1.0) indicating how well the job description matches the role
2. Brief reasoning for your confidence score

Return your response as a JSON object with the following structure:
{{
  "evaluations": [
    {{
      "role": "Role Name",
      "confidence": 0.85,
      "reasoning": "Brief explanation"
    }}
  ]
}}

Evaluate ALL roles listed above."""
    
    return prompt

print("✅ Role confidence prompt builder defined")


In [ ]:
# ==== Cell 15.2 — Role Confidence Shortlist (robust build + canonicalize + write; empty-safe) =====
import json
from openai import OpenAI

client = OpenAI()

def call_llm_json(prompt: str, model: str = None) -> dict:
    """Call OpenAI API with JSON response."""
    if model is None:
        model = MODEL_ID
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.2
    )
    
    return json.loads(response.choices[0].message.content)

# Get MNPS roles and KSACs
VALID_ROLES = roles_df['Role'].tolist()
KSACS_TEXT = """[KSACs content from the roles file]"""

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")
print(f"✅ Using model: {MODEL_ID}")


In [ ]:
# ==== Cell 16 — Batch Processing with GPT-5-pro =====
import time
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""
    
    # Get role confidence
    prompt = build_role_confidence_prompt(job_text, VALID_ROLES, KSACS_TEXT)
    
    try:
        response = call_llm_json(prompt, MODEL_ID)
        evaluations = response.get('evaluations', [])
        
        # Find best role
        best_role = max(evaluations, key=lambda x: x['confidence']) if evaluations else None
        
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': f"{best_role['role']} I" if best_role else 'Unknown',
            'major_role_group': best_role['role'] if best_role else 'Other',
            'minor_sub_group': 'I',  # Default to I, will be refined in v7.5 corrections
            'grouping_justification': best_role['reasoning'] if best_role else 'No match found',
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.1)  # Rate limiting

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")


## **5** | v7.5 Corrections Applied

Now apply the same corrections from v7.5 to the GPT-5-pro results:


In [ ]:
# ==== v7.5 Corrections for GPT-5-pro Results =====
import re
import numpy as np

# Load the GPT-5-pro results
preds = results_df.copy()
attrs = df.copy()

# Closed sets and normalization helpers
MAJOR_ALLOWED = [
    'Technician','Specialist','Analyst','Manager','Coordinator','Director','Other',
    'Teacher','Coach','Counselor','Clerical Support','Instructor','Driver'
]
MINOR_ALLOWED = ['I','II','III','Lead']

CANON_MINOR_MAP = {
    'i':'I','1':'I','one':'I','entry':'I',
    'ii':'II','2':'II','two':'II',
    'iii':'III','3':'III','three':'III',
    'lead':'Lead','iv':'III','4':'III'
}

SPECIALIST_FALLBACKS = [
    ('Teacher','classroom|lesson|instruction|teacher|students'),
    ('Coach','coach|instructional coach|plc|model lessons|co-teach'),
    ('Clerical Support','clerk|clerical|records|data entry|office support'),
    ('Counselor','counsel|social-emotional|guidance'),
    ('Manager','manage|supervise|budget|oversight|lead team|program manager'),
]

def normalize_minor(x: str) -> str:
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

print("✅ v7.5 correction functions defined")


In [ ]:
# ==== Apply v7.5 Corrections =====

# Build attribute-only text
ATTR_COLS = [
    'Position Summary','Essential Functions','Work Experience','Education',
    'Licenses and Certifications','Knowledge, Skills and Abilities'
]

text = (
    attrs['Position Summary'].fillna('') + ' ' +
    attrs['Essential Functions'].fillna('') + ' ' +
    attrs['Work Experience'].fillna('') + ' ' +
    attrs['Education'].fillna('') + ' ' +
    attrs['Licenses and Certifications'].fillna('') + ' ' +
    attrs['Knowledge, Skills and Abilities'].fillna('')
)

# Apply corrections
maj0 = preds.get('major_role_group', pd.Series(['Other']*len(preds)))
min0 = preds.get('minor_sub_group', pd.Series(['I']*len(preds)))

ref_major = []
for i, m in enumerate(maj0):
    proposed = str(m) if pd.notna(m) else 'Other'
    proposed = proposed if proposed in MAJOR_ALLOWED else 'Other'
    proposed = discourage_specialist(text.iloc[i], proposed)
    ref_major.append(proposed)

ref_minor = [normalize_minor(x) for x in min0]

# Create corrected results
corrected = preds.copy()
corrected['major_role_group'] = ref_major
corrected['minor_sub_group'] = ref_minor

# Save corrected results
corrected_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro_v75_corrected.csv"
corrected.to_csv(corrected_path, index=False)

print(f"✅ Applied v7.5 corrections")
print(f"✅ Saved corrected results to: {corrected_path}")


In [ ]:
# ==== Generate Summary Statistics =====

before_major = preds.get('major_role_group', pd.Series(['']*len(preds))).astype(str)
before_minor = preds.get('minor_sub_group', pd.Series(['']*len(preds))).astype(str)
after_major  = corrected['major_role_group'].astype(str)
after_minor  = corrected['minor_sub_group'].astype(str)

counts = pd.DataFrame({
    'key': ['rows','major_changed','minor_changed','specialist_after_count'],
    'value': [
        len(corrected),
        int((before_major!=after_major).sum()),
        int((before_minor!=after_minor).sum()),
        int((after_major=='Specialist').sum())
    ]
})

counts_path = OUTPUTS_DIR / "correction_counts_gpt5pro.csv"
counts.to_csv(counts_path, index=False)

# Show examples of changes
ex_idx = ((before_major!=after_major) | (before_minor!=after_minor)).to_numpy().nonzero()[0][:6]
examples = pd.DataFrame({
    'row': ex_idx,
    'job_title_original': preds['job_title_original'].iloc[ex_idx],
    'major_before': before_major.iloc[ex_idx],
    'major_after': after_major.iloc[ex_idx],
    'minor_before': before_minor.iloc[ex_idx],
    'minor_after': after_minor.iloc[ex_idx],
})

examples_path = OUTPUTS_DIR / "examples_gpt5pro.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(counts.to_string(index=False))

print("\n📝 Example Corrections:")
print(examples.to_string(index=False))

print(f"\n✅ Saved counts to: {counts_path}")
print(f"✅ Saved examples to: {examples_path}")


: 

d